In [1]:
import os
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt

import pipeline_config as pconf
from comparison_check_log_utils import (
    resolve_final_directory,
    nearest_final_spectrum_native,
    index_final_native_files,
    collect_input_spectra_for_mode,
    resolve_comparison_spectrum_path,
    load_comparison_spectrum_xy,
)


In [2]:
import pipeline_config as pconf
from pipeline_config import bootstrap_runtime

rt = pconf.bootstrap_runtime(photometry_stage="extrapolated")
OUTPUT_DIR = rt.output_dir
OUTPUT_PATH = rt.output_path
DATASPEC_PATH = rt.dataspec_path
DATAINFO_PATH = rt.datainfo_path
FILTER_PATH = rt.filter_path
FILTER_LEAF = rt.filter_leaf
FILTERS_PARENT = rt.filters_parent
color_dict, mark_dict, exclude_filt = rt.color_dict, rt.mark_dict, rt.exclude_filt
FINAL_SPECTRA_DIR = rt.final_spectra_dir
GP_MODE = rt.gp_mode
import what_the_flux as wtf

    import comparison_check_log_utils as cc


### Resolved FINAL directories (spliced / full_gp)

In [ ]:
def _final_dir(product: str) -> str:
    if product == pconf.SUBDIR_FULL_GP:
        return FINAL_SPECTRA_DIR
    return pconf.twodim_iter_final_spectra_dir(pconf.outputs_root(COCO_PATH), SNNAME, None, product=product)

dir_spliced = _final_dir(pconf.SUBDIR_SPLICED)
dir_full_gp = _final_dir(pconf.SUBDIR_FULL_GP)
print("spliced:", dir_spliced)
print("full_gp:", dir_full_gp)


### Core plotting

`plot_native_epoch` (single definition in the code cell below) uses `nearest_final_spectrum_native` per SED directory. Effective query MJD: ``t0_mjd + phase_query_days`` if ``phase_query_days is not None``, else ``mjd_query``, else ``spec_mjd_fallback`` (batch).

The plot title and legend title use the **matched** SED epoch (days), not the raw query. Optional: ``plot_inputs`` (SED-only), colors, axis limits, ``savefig_path``.


In [4]:
def effective_query_mjd(
    *,
    phase_query_days,
    mjd_query,
    t0_mjd,
    spec_mjd_fallback=None,
):
    if phase_query_days is not None:
        return float(t0_mjd) + float(phase_query_days)
    if mjd_query is not None:
        return float(mjd_query)
    if spec_mjd_fallback is not None:
        return float(spec_mjd_fallback)
    raise ValueError(
        "Set phase_query_days, mjd_query, or spec_mjd_fallback (batch) to choose an epoch."
    )


def plot_native_epoch(
    *,
    phase_query_days=None,
    mjd_query=None,
    t0_mjd=None,
    spec_mjd_fallback=None,
    which_sed="spliced",
    dir_spliced=None,
    dir_full_gp=None,
    mode="smoothed",
    list_file=None,
    original_spec_dir=None,
    time_window=0.0,
    z=0.0,
    sed_scale=1.0,
    input_scale=1.0,
    normalize_median=False,
    plot_inputs=True,
    sed_color="black",
    sed_colors=None,
    input_colors=None,
    xlim=None,
    ylim=None,
    savefig_path=None,
    savefig_dpi=600,
    ax=None,
    figsize=(9, 5.5),
    show=True,
    title_prefix=None,
):
    """Plot native FINAL (spliced and/or full_gp) vs optional input spectra.

    which_sed: "spliced" | "full_gp" | "both"

    SED curves use ``sed_scale`` only (not median-normalized). When
    ``normalize_median`` is True, each input spectrum is rescaled as
    ``(orig / nanmedian(orig)) * nanmedian(SED)`` where the SED median is taken
    on pixels overlapping the input wavelength range (first plotted SED).

    ``plot_inputs=False``: SED only. ``sed_color`` / ``sed_colors`` control SED
    line colors; ``input_colors`` cycles colors for inputs. Matched SED epoch
    (days) goes in the plot title and legend title, not in the SED legend entry.
    """
    import matplotlib.pyplot as plt
    from matplotlib.ticker import AutoMinorLocator

    if t0_mjd is None:
        t0_mjd = float(pconf.SN_EXPLOSION_MJD.get(SNNAME, np.nan))
    q_mjd = effective_query_mjd(
        phase_query_days=phase_query_days,
        mjd_query=mjd_query,
        t0_mjd=t0_mjd,
        spec_mjd_fallback=spec_mjd_fallback,
    )

    branches = []
    which_sed = str(which_sed).lower()
    if which_sed not in ("spliced", "full_gp", "both"):
        raise ValueError("which_sed must be spliced, full_gp, or both")
    if which_sed in ("spliced", "both"):
        branches.append(("spliced", dir_spliced))
    if which_sed in ("full_gp", "both"):
        branches.append(("full_gp", dir_full_gp))

    plt.rcParams["font.family"] = "serif"
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    ax.axvspan(13125, 14375, color='lightgray', alpha=0.5)
    ax.axvspan(17812.5, 19375, color='lightgray', alpha=0.5)
    ax.axvspan(9800, 10100, color='lightgray', alpha=0.5)
    ax.axvspan(5300, 5700, color='lightgray', alpha=0.5)

    color_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    ref_sed_wl = None
    ref_sed_fl = None
    phase_matched_sed = None

    for j, (branch_tag, fdir) in enumerate(branches):
        if not fdir:
            warnings.warn("skip %s: no directory" % branch_tag)
            continue
        try:
            smjd, sed_wl, sed_fl, sed_fname = nearest_final_spectrum_native(
                fdir,
                q_mjd,
                COCO_PATH,
                SNNAME,
                flux_on_disk=FINAL_FLUX_ON_DISK,
                datalc_path=DATALC_PATH,
                final_suffixes=FINAL_SUFFIXES_TO_LOAD,
            )
        except Exception as e:
            warnings.warn("nearest FINAL failed for %s: %s" % (branch_tag, e))
            continue

        sed_fl = sed_scale * np.asarray(sed_fl, dtype=float)
        sed_wl = np.asarray(sed_wl, dtype=float)

        if ref_sed_wl is None:
            ref_sed_wl = sed_wl.copy()
            ref_sed_fl = sed_fl.copy()
            phase_matched_sed = float(smjd) - float(t0_mjd)

        if sed_colors is not None:
            c = sed_colors[j % len(sed_colors)]
        elif which_sed == "both":
            c = color_cycle[j % len(color_cycle)]
        else:
            c = sed_color

        ls = "-" if j == 0 else "--"
        if which_sed == "both":
            ls = "-" if j == 0 else "--"

        ax.plot(
            sed_wl,
            sed_fl,
            color=c,
            ls=ls,
            lw=2,
            label="SED",
            alpha=0.9,
        )

    if phase_matched_sed is None:
        phase_matched_sed = float(q_mjd) - float(t0_mjd)
        warnings.warn(
            "No SED curve plotted; using query phase %.4f d for title/legend."
            % phase_matched_sed
        )

    if plot_inputs:
        mode = str(mode).lower()
        orig_paths, orig_mjds, _odir, list_ref = collect_input_spectra_for_mode(
            mode, list_file, original_spec_dir, SNNAME, COCO_PATH
        )

        if not orig_paths or orig_mjds.size == 0:
            warnings.warn(
                "No input spectra (mode=%r list_ref=%r)" % (mode, list_ref)
            )
            idxs = np.array([], dtype=int)
        elif time_window and time_window > 0:
            idxs = np.where(np.abs(orig_mjds - q_mjd) <= time_window)[0]
            if idxs.size == 0:
                idxs = np.array([int(np.argmin(np.abs(orig_mjds - q_mjd)))])
        else:
            idxs = np.array([int(np.argmin(np.abs(orig_mjds - q_mjd)))])

        fdisk = FINAL_FLUX_ON_DISK if mode == "mangled" else "auto"

        for k, idx in enumerate(idxs):
            orig_path = orig_paths[int(idx)]
            orig_mjd = float(orig_mjds[int(idx)])
            disk_path = resolve_comparison_spectrum_path(
                orig_path, mode, _odir, COCO_PATH, SNNAME
            )
            try:
                orig_wl, orig_flux = load_comparison_spectrum_xy(
                    disk_path, mode, fdisk
                )
            except Exception as e:
                warnings.warn("Could not load %s: %s" % (disk_path, e))
                continue

            orig_wl = np.asarray(orig_wl, dtype=float)
            orig_flux = input_scale * np.asarray(orig_flux, dtype=float)

            if z > 0.0:
                orig_wl = orig_wl / (1.0 + z)
                orig_flux = orig_flux * (1.0 + z)

            if normalize_median:
                m_comp = np.isfinite(orig_flux)
                if np.any(m_comp) and ref_sed_wl is not None:
                    comp_med = float(np.nanmedian(orig_flux[m_comp]))

                    if np.isfinite(comp_med) and comp_med > 0.0:
                        min_wl = np.nanmin(orig_wl[m_comp])
                        max_wl = np.nanmax(orig_wl[m_comp])

                        m_sed_overlap = (
                            (ref_sed_wl >= min_wl)
                            & (ref_sed_wl <= max_wl)
                            & np.isfinite(ref_sed_fl)
                        )

                        if np.any(m_sed_overlap):
                            sed_med = float(np.nanmedian(ref_sed_fl[m_sed_overlap]))
                            if np.isfinite(sed_med):
                                orig_flux = (
                                    orig_flux / comp_med
                                ) * sed_med
                            else:
                                warnings.warn(
                                    "SED overlap median not finite for %s" % disk_path
                                )
                        else:
                            warnings.warn(
                                "No overlapping SED region for %s" % disk_path
                            )
                    else:
                        warnings.warn(
                            "Comparison median invalid for %s" % disk_path
                        )

            orig_phase = orig_mjd - float(t0_mjd)
            if input_colors is not None:
                ic = input_colors[k % len(input_colors)]
            else:
                ic = color_cycle[
                    (k + len(branches)) % len(color_cycle)
                ]

            ax.plot(
                orig_wl,
                orig_flux,
                label="t=%.2f days" % orig_phase,
                color=ic,
                alpha=0.9,
                lw=1.5,
            )
    
    ax.set_xlabel("Wavelength (Å)", fontsize=16)
    ax.set_ylabel(r"Flux (erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$)", fontsize=16)

    title_parts = []
    if title_prefix:
        title_parts.append(str(title_prefix))
    title_parts.append("Matched SED epoch $t = %.2f$ d" % phase_matched_sed)
    #ax.set_title(" — ".join(title_parts))

    # leg = ax.legend(
    #     title="Phase: t=%.2f days" % phase_matched_sed,
    #     fontsize=14,
    #     loc="upper left",
    # )
    # if leg is not None:
    #     leg.get_title().set_fontsize(14)
    
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, which="major", alpha=0.5)
    ax.grid(True, which="minor", alpha=0.2, linestyle=":")

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    #fig.tight_layout()

    if savefig_path:
        fig.savefig(savefig_path, dpi=float(savefig_dpi))

    if show:
        plt.show()
    return fig, ax, q_mjd


def union_spec_mjds_for_batch(which_sed, dir_spliced, dir_full_gp):
    """Sorted unique spec_mjd from indexed FINAL trees (lightweight)."""
    which_sed = str(which_sed).lower()
    out = []
    if which_sed in ("spliced", "both"):
        out.extend(
            index_final_native_files(
                dir_spliced,
                COCO_PATH,
                SNNAME,
                datalc_path=DATALC_PATH,
                final_suffixes=FINAL_SUFFIXES_TO_LOAD,
            )
        )
    if which_sed in ("full_gp", "both"):
        out.extend(
            index_final_native_files(
                dir_full_gp,
                COCO_PATH,
                SNNAME,
                datalc_path=DATALC_PATH,
                final_suffixes=FINAL_SUFFIXES_TO_LOAD,
            )
        )
    mjds = sorted({float(t[0]) for t in out})
    return mjds


### Interactive: one epoch

In [ ]:
# Query: phase_query_days wins over mjd_query when not None
phase_query_days = 1  # days since t0_mjd; set to None to use mjd_query instead
mjd_query = None  # observer MJD, e.g. t0_mjd + 1.0

which_sed = "full_gp"  # "spliced" | "full_gp" | "both"
mode = "mangled"
time_window = 0.5
normalize_median = True
sed_scale = 1.0
input_scale = 1.0

# Optional styling / I/O (see plot_native_epoch docstring):
plot_inputs = True
sed_color = "black"
figsize = (8, 4.5)
# sed_colors = ("C0", "C3")  # when which_sed="both"
input_colors = ["tomato", "deepskyblue", "gold", 'limegreen', 'hotpink']
# xlim, ylim = (3000, 25000), None
#ylim = (0, 0.5e-19)
#savefig_path = '/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/spectra_slice_5days.png'

plot_native_epoch(
    phase_query_days=phase_query_days,
    mjd_query=mjd_query,
    t0_mjd=t0_mjd,
    which_sed=which_sed,
    dir_spliced=dir_spliced,
    dir_full_gp=dir_full_gp,
    z=z_redshift,
    sed_scale=sed_scale,
    show=True,
    plot_inputs=plot_inputs,
    input_colors=input_colors,
    sed_color=sed_color,
    figsize=figsize,
    mode=mode,
    time_window=time_window,
    normalize_median=normalize_median,
    input_scale=input_scale,
    #ylim=ylim,
    #savefig_path=savefig_path,
)


In [ ]:
# `plot_native_epoch`, `effective_query_mjd`, and `union_spec_mjds_for_batch` are defined in the **Core plotting** cell above.
# Run that cell before demos / batch below. (This cell previously duplicated `plot_native_epoch`; removed to avoid overriding.)


In [ ]:
# Query: phase_query_days wins over mjd_query when not None
phase_query_days = 5.0  # days since t0_mjd; set to None to use mjd_query instead
mjd_query = None  # observer MJD, e.g. t0_mjd + 1.0

which_sed = "full_gp"  # "spliced" | "full_gp" | "both"
mode = "mangled"
time_window = 0.20
normalize_median = True
sed_scale = 1.0
input_scale = 1.0

# Optional styling / I/O (see plot_native_epoch docstring):
# plot_inputs = True
# sed_color = "black"
# sed_colors = ("C0", "C3")  # when which_sed="both"
# input_colors = ["tab:orange", "tab:green", "tab:red"]
# xlim, ylim = (3000, 25000), None
# savefig_path = os.path.join(COCO_PATH, "Outputs", SNNAME, "native_epoch.png")

plot_native_epoch(
    phase_query_days=phase_query_days,
    mjd_query=mjd_query,
    t0_mjd=t0_mjd,
    which_sed=which_sed,
    dir_spliced=dir_spliced,
    dir_full_gp=dir_full_gp,
    #mode=mode,
    time_window=time_window,
    z=z_redshift,
    sed_scale=sed_scale,
    input_scale=input_scale,
    normalize_median=normalize_median,
    show=True,
)

### Batch save PNGs (absolute flux)

In [ ]:
SAVE_ROOT = os.path.join(
    COCO_PATH, "Outputs", SNNAME, "spectra_compare_native", RUN_LABEL, "absolute"
)
os.makedirs(SAVE_ROOT, exist_ok=True)

batch_which_sed = "spliced"
batch_mode = "smoothed"
batch_time_window = 0.5
batch_normalize_median = False

mjds = union_spec_mjds_for_batch(batch_which_sed, dir_spliced, dir_full_gp)
print("epochs:", len(mjds))

for spec_mjd in mjds:
    fig, ax, q = plot_native_epoch(
        mjd_query=float(spec_mjd),
        t0_mjd=t0_mjd,
        phase_query_days=None,
        which_sed=batch_which_sed,
        dir_spliced=dir_spliced,
        dir_full_gp=dir_full_gp,
        mode=batch_mode,
        time_window=batch_time_window,
        z=z_redshift,
        normalize_median=batch_normalize_median,
        show=False,
    )
    fn = "native_%s_which=%s_mjd=%.6f.png" % (
        SNNAME, batch_which_sed, float(spec_mjd)
    )
    fig.savefig(os.path.join(SAVE_ROOT, fn), dpi=150, bbox_inches="tight")
    plt.close(fig)
print("wrote under", SAVE_ROOT)


### Batch save PNGs (median-normalized)

In [ ]:
SAVE_ROOT_NORM = os.path.join(
    COCO_PATH, "Outputs", SNNAME, "spectra_compare_native", RUN_LABEL, "median_norm"
)
os.makedirs(SAVE_ROOT_NORM, exist_ok=True)

batch_which_sed = "full_gp"
batch_mode = "mangled"
batch_time_window = 0.5
batch_normalize_median = True

mjds = union_spec_mjds_for_batch(batch_which_sed, dir_spliced, dir_full_gp)
for spec_mjd in mjds:
    fig, ax, q = plot_native_epoch(
        mjd_query=float(spec_mjd),
        t0_mjd=t0_mjd,
        which_sed=batch_which_sed,
        dir_spliced=dir_spliced,
        dir_full_gp=dir_full_gp,
        mode=batch_mode,
        time_window=batch_time_window,
        z=z_redshift,
        normalize_median=batch_normalize_median,
        show=False,
    )
    fn = "native_median_%s_which=%s_mjd=%.6f.png" % (
        SNNAME, batch_which_sed, float(spec_mjd)
    )
    fig.savefig(os.path.join(SAVE_ROOT_NORM, fn), dpi=150, bbox_inches="tight")
    plt.close(fig)
print("wrote under", SAVE_ROOT_NORM)


### Phase vs MJD sanity check

For the same `t0_mjd`, `phase_query_days=1.0` should match `mjd_query=t0_mjd+1` up to the nearest-FINAL epoch mapping.


In [ ]:
fig1, ax1, q1 = plot_native_epoch(
    phase_query_days=1.0,
    mjd_query=None,
    t0_mjd=t0_mjd,
    which_sed="spliced",
    dir_spliced=dir_spliced,
    dir_full_gp=dir_full_gp,
    mode="smoothed",
    show=False,
    title_prefix="A: phase=1d",
)
fig2, ax2, q2 = plot_native_epoch(
    phase_query_days=None,
    mjd_query=float(t0_mjd) + 1.0,
    t0_mjd=t0_mjd,
    which_sed="spliced",
    dir_spliced=dir_spliced,
    dir_full_gp=dir_full_gp,
    mode="smoothed",
    show=False,
    title_prefix="B: mjd=t0+1",
)
assert abs(q1 - q2) < 1e-9
plt.close(fig1)
plt.close(fig2)
print("phase vs MJD query consistency OK (q = %.10f)" % q1)


# make movie

In [22]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

def create_spectra_evolution_movie(
    *,
    dir_spliced,
    dir_full_gp,
    t0_mjd,
    which_sed="spliced",
    xlim=(3000, 250000),  # Set your fixed X limits
    ylim=(0, 8e-16),     # Set your fixed Y limits
    output_filename="spectra_evolution_ryan.mp4",
    fps=6,
    **plot_kwargs
):
    """
    Generates a movie looping through all available spectra.
    """
    # 1. Fetch all unique MJDs from your directory
    all_mjds = union_spec_mjds_for_batch(which_sed, dir_spliced, dir_full_gp)
    
    if not all_mjds:
        raise ValueError("No spectra found to animate.")

    # 2. Initialize the figure
    fig, ax = plt.subplots(figsize=(9, 5.5))
    # Force a permanent 15% margin on the left side to fit the log labels
    fig.subplots_adjust(left=0.20)

    # 3. Define the update function for each frame
# 3. Define the update function for each frame
    def update(frame_idx):
        ax.clear()  # Clear the previous frame
        current_mjd = all_mjds[frame_idx]

        # Call your existing function
        plot_native_epoch(
            mjd_query=current_mjd,
            t0_mjd=t0_mjd,
            which_sed=which_sed,
            dir_spliced=dir_spliced,
            dir_full_gp=dir_full_gp,
            plot_inputs=False, 
            xlim=xlim,
            ylim=ylim,
            ax=ax,
            show=False,
            **plot_kwargs
        )

        # Apply the log scale here, every frame!
        ax.set_yscale('log')

        # 4. Add the prominent timer text
        phase_days = current_mjd - float(t0_mjd)
        ax.text(
            0.05, 0.90, 
            f't = {phase_days:+.2f} days', 
            transform=ax.transAxes, 
            fontsize=18, 
            fontweight='bold',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray', boxstyle='round,pad=0.5')
        )

    # 5. Create and save the animation
    anim = FuncAnimation(fig, update, frames=len(all_mjds), interval=1000/fps)
    
    # Requires ffmpeg installed on your system
    anim.save(output_filename, fps=fps, extra_args=['-vcodec', 'libx264'])
    #anim.save(output_filename, fps=fps)
    
    # Close the figure so it doesn't display statically in a notebook/console
    plt.close(fig)

In [23]:
import os

# 1. Define your fixed axes limits (crucial for a stable video)
which_sed = 'full_gp'
fixed_xlim = (3000, 25000)  # Using the xlim from your snippet
fixed_ylim = (2e-20, 1e-15)     # CHANGE THIS: Set to the actual min/max of your flux data

# 2. Define the output path
# movie_out_path = os.path.join(COCO_PATH, "Outputs", SNNAME, "sed_evolution.mp4")
# Change the file extension from .mp4 to .gif
movie_out_path = os.path.join(COCO_PATH, "Outputs", SNNAME, "sed_evolution_ryan_logflux.mp4")

# 3. Call the movie function
create_spectra_evolution_movie(
    dir_spliced=dir_spliced,
    dir_full_gp=dir_full_gp,
    t0_mjd=t0_mjd,
    which_sed=which_sed,          # Currently set to "full_gp"
    xlim=fixed_xlim,
    ylim=fixed_ylim,
    output_filename=movie_out_path,
    fps=5,                        # Adjust frames-per-second to speed up/slow down
    # Pass-through kwargs for the plot_native_epoch function:
    z=z_redshift,
    sed_scale=sed_scale,
    sed_color="deepskyblue"             # Optional: keep consistent styling
)

In [ ]:
# Example: GP training points on top of FINAL (same knobs as plot_native_epoch)
#mask_telluric_rows=False
plot_native_epoch_with_gp_scaled(
    phase_query_days=7.5,
    which_sed="full_gp",
    dir_spliced=dir_spliced,
    dir_full_gp=dir_full_gp,
    plot_inputs=False,
    gp_overlap_arm_scale=True,
    normalize_median=True,
    gp_colors=["deepskyblue", "hotpink", "limegreen", "gold", "tomato"],
    savefig_path='/Users/ravkaur/Desktop/research/kilonova-SED/kn-sed-pipeline/Outputs/AT2017gfo/plots-for-flash/newgp_spectra_7.5days.png',
    #gp_mask_telluric_rows=True,
    ylim=(0, 3e-17),
)
